# Assignment 4

本次作业涵盖了用于全景拼接的Harris角点检测器、RANSAC算法以及HOG描述符

In [ ]:
# Setup
import numpy as np
from skimage.feature import corner_peaks
from skimage.io import imread
import matplotlib.pyplot as plt
from skimage import filters
from skimage.feature import corner_peaks
from skimage.util.shape import view_as_blocks
from scipy.spatial.distance import cdist
from scipy.ndimage.filters import convolve

from utils import pad, unpad, get_output_space, warp_image


%matplotlib inline
plt.rcParams['figure.figsize'] = (15.0, 12.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading extenrnal modules
%load_ext autoreload
%autoreload 2

## 全景拼接
全景拼接是计算机视觉领域早期的成功案例之一。2007年，Matthew Brown 和 David G. Lowe 发表了一篇著名的[全景图像拼接论文](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf)。此后，自动全景拼接技术被广泛应用于许多场景，例如谷歌街景、智能手机上的全景照片，以及 Photosynth 和 AutoStitch 等拼接软件中。

在本作业中，我们将检测并匹配多幅图像中的关键点，从而构建出一张完整的全景图像。这涉及以下几个任务：
1. 使用 Harris 角点检测器查找关键点。
2. 构建描述子来描述图像中的每个点。<br>
   比较来自两幅不同图像的两组描述子，并找出匹配的关键点。
3. 给定匹配关键点列表，使用最小二乘法求出将一幅图像中的点映射到另一幅图像的仿射变换矩阵。
4. 使用 RANSAC 算法对仿射变换矩阵进行更鲁棒的估计。<br>
   在得到变换矩阵后，用它来变换第二幅图像，并将其叠加到第一幅图像上，从而生成全景图。
5. 实现另一种不同的描述子（HOG 描述子），并得到另一个拼接结果。

## 1.Harris 角点检测器

在本节中，你将实现用于关键点定位的 Harris 角点检测器。请回顾课堂讲义中关于 Harris 角点检测器的内容，以理解其工作原理。Harris 角点检测算法可以分为以下几个步骤：

1. 计算图像的 $x$ 和 $y$ 方向导数（$I_x, I_y$）
2. 计算每个像素处的导数乘积（$I_x^2, I_y^2, I_{xy}$）
3. 计算每个像素处的矩阵 $M$，其中

$$
M = \sum_{x,y} w(x,y)
    \begin{bmatrix}
        I_{x}^2 & I_{x}I_{y} \\
        I_{x}I_{y} & I_{y}^2
    \end{bmatrix}
$$

4. 计算每个像素处的角点响应 $R = \text{Det}(M) - k(\text{Trace}(M)^2)$
5. 输出角点响应图 $R(x,y)$


*-提示：建议使用 `scipy.ndimage.filters.convolve` 函数*

In [ ]:
def harris_corners(img, window_size=3, k=0.04):
    """
    Compute Harris corner response map. Follow the math equation
    R=Det(M)-k(Trace(M)^2).

    Hint:
        You may use the function scipy.ndimage.filters.convolve,
        which is already imported above.

    Args:
        img: Grayscale image of shape (H, W)
        window_size: size of the window function
        k: sensitivity parameter

    Returns:
        response: Harris response image of shape (H, W)
    """

    H, W = img.shape
    window = np.ones((window_size, window_size))

    response = np.zeros((H, W))

    dx = filters.sobel_v(img)
    dy = filters.sobel_h(img)

    ### YOUR CODE HERE

    ### END YOUR CODE

    return response

In [ ]:
img = imread('sudoku.png', as_gray=True)

# Compute Harris corner response
response = harris_corners(img)


np.save('harris_response.npy', response)

# Display corner response
plt.imshow(response)
plt.axis('off')
plt.title('Harris Corner Response')


plt.show()

正确实现 Harris 检测器后，你将能够在输出的角点响应图像中看到数独网格和字母的角点周围出现小小的亮斑。`skimage.feature` 模块中的 `corner_peaks` 函数会执行非极大值抑制，提取响应图中的局部极大值，从而定位出关键点。

In [ ]:
# Perform non-maximum suppression in response map
# and output corner coordiantes
corners = corner_peaks(response, threshold_rel=0.01)

# Display detected corners
plt.imshow(img)
plt.scatter(corners[:,1], corners[:,0], marker='x')
plt.axis('off')
plt.title('Detected Corners')
plt.show()

## 2. 描述与匹配关键点

现在，我们已经能够通过在两张图像上分别运行 Harris 角点检测器来定位各自的关键点。接下来的问题是：我们如何确定哪些关键点对来自这两张图像中的对应位置？为了*匹配*检测到的关键点，我们必须提出一种基于关键点局部外观来*描述*它们的方法。通常，每个检测到的关键点周围的局部区域会被转换为一个固定大小的向量，称为*描述子*。

### 2.1 创建描述子 方向梯度直方图 (HOG)

而在本节中，你将实现一个简化版的 HOG 描述子。<br>
HOG 代表方向梯度直方图（Histogram of Oriented Gradients）。在 HOG 描述子中，梯度方向（即方向梯度）的分布（直方图）被用作特征。图像的梯度（$x$ 和 $y$ 方向导数）非常有用，因为在边缘和角点（灰度剧烈变化的区域）附近，梯度的幅值较大，而且我们知道，相比于平坦区域，边缘和角点包含了更多关于物体形状的信息。<br>
HOG 的步骤如下：
    1. 计算图像在 $x$ 和 $y$ 方向的梯度图
        使用 `skimage.filters` 提供的 Sobel 滤波器
    2. 计算梯度直方图
        将图像划分为若干个细胞（cell），并在每个细胞中计算梯度的直方图
    3. 将每个块（block）中的直方图展平为特征向量
    4. 对展平后的块进行归一化

实现 **`hog_descriptor`** 函数。

In [ ]:
def hog_descriptor(patch, pixels_per_cell=(8,8)):
    """
    Ref: http://lear.inrialpes.fr/people/triggs/pubs/Dalal-cvpr05.pdf
    
    Generating hog descriptor by the following steps:

    1. Compute the gradient image in x and y directions (already done for you)
    2. Compute gradient histograms for each cell
    3. Flatten block of histograms into a 1D feature vector
        Here, we treat the entire patch of histograms as our block
    4. Normalize flattened block
        Normalization makes the descriptor more robust to lighting variations

    Args:
        patch: grayscale image patch of shape (H, W)
        pixels_per_cell: size of a cell with shape (M, N)

    Returns:
        block: 1D patch descriptor array of shape ((H*W*n_bins)/(M*N))
    """
    assert (patch.shape[0] % pixels_per_cell[0] == 0),\
                'Heights of patch and cell do not match'
    assert (patch.shape[1] % pixels_per_cell[1] == 0),\
                'Widths of patch and cell do not match'

    n_bins = 9
    degrees_per_bin = 180 // n_bins

    Gx = filters.sobel_v(patch)
    Gy = filters.sobel_h(patch)

    # Unsigned gradients
    G = np.sqrt(Gx**2 + Gy**2)
    theta = (np.arctan2(Gy, Gx) * 180 / np.pi) % 180

    # Group entries of G and theta into cells of shape pixels_per_cell, (M, N)
    #   G_cells.shape = theta_cells.shape = (H//M, W//N)
    #   G_cells[0, 0].shape = theta_cells[0, 0].shape = (M, N)
    G_cells = view_as_blocks(G, block_shape=pixels_per_cell)
    theta_cells = view_as_blocks(theta, block_shape=pixels_per_cell)
    rows = G_cells.shape[0]
    cols = G_cells.shape[1]

    # For each cell, keep track of gradient histrogram of size n_bins
    cells = np.zeros((rows, cols, n_bins))

    # Compute histogram per cell
    ### YOUR CODE HERE

    ### YOUR CODE HERE

    return block

In [ ]:
img1 = imread('yosemite1.jpg', as_gray=True)
img2 = imread('yosemite2.jpg', as_gray=True)

# Detect keypoints in two images
keypoints1 = corner_peaks(harris_corners(img1, window_size=3),
                          threshold_rel=0.05,
                          exclude_border=8)
keypoints2 = corner_peaks(harris_corners(img2, window_size=3),
                          threshold_rel=0.05,
                          exclude_border=8)

# Display detected keypoints

img1_color = imread('yosemite1.jpg')
img2_color = imread('yosemite2.jpg')

print(keypoints1.shape)
tmp = np.load('./references/keypoints1.npy')
print(tmp.shape)


plt.subplot(1,2,1)
plt.imshow(img1_color)
plt.scatter(keypoints1[:,1], keypoints1[:,0], marker='x')
plt.axis('off')
plt.title('Detected Keypoints for Image 1')

plt.subplot(1,2,2)
plt.imshow(img2_color)
plt.scatter(keypoints2[:,1], keypoints2[:,0], marker='x')
plt.axis('off')
plt.title('Detected Keypoints for Image 2')
plt.show()

### 2.2 匹配描述子

接下来，实现 **`match_descriptors`** 函数，以在两组描述子之间找到好的匹配。首先，计算图像1和图像2中所有描述子对之间的欧氏距离。然后，利用这些距离判断是否存在好的匹配：如果最近向量的距离显著小于（按给定的比率）第二近向量的距离，则将其视为一个匹配。函数的输出是一个数组，其中每一行包含一对匹配描述子的索引。

In [ ]:
def describe_keypoints(image, keypoints, desc_func, patch_size=16):
    """
    Args:
        image: grayscale image of shape (H, W)
        keypoints: 2D array containing a keypoint (y, x) in each row
        desc_func: function that takes in an image patch and outputs
            a 1D feature vector describing the patch
        patch_size: size of a square patch at each keypoint

    Returns:
        desc: array of features describing the keypoints
    """

    image.astype(np.float32)
    desc = []

    for i, kp in enumerate(keypoints):
        y, x = kp
        patch = image[y-(patch_size//2):y+((patch_size+1)//2),
                      x-(patch_size//2):x+((patch_size+1)//2)]
        desc.append(desc_func(patch))
    return np.array(desc)

In [ ]:
def match_descriptors(desc1, desc2, threshold=0.5):
    """
    Match the feature descriptors by finding distances between them. A match is formed
    when the distance to the closest vector is much smaller than the distance to the
    second-closest, that is, the ratio of the distances should be smaller
    than the threshold. Return the matches as pairs of vector indices.

    Hint:
        The Numpy functions np.sort, np.argmin, np.asarray might be useful

    Args:
        desc1: an array of shape (M, P) holding descriptors of size P about M keypoints
        desc2: an array of shape (N, P) holding descriptors of size P about N keypoints

    Returns:
        matches: an array of shape (Q, 2) where each row holds the indices of one pair
        of matching descriptors
    """
    matches = []

    N = desc1.shape[0]
    dists = cdist(desc1, desc2)

    ### YOUR CODE HERE

    ### END YOUR CODE

    return matches

In [ ]:
from utils import plot_matches

patch_size = 16
    
# Extract features from the corners
desc1 = describe_keypoints(img1, keypoints1,
                           desc_func=hog_descriptor,
                           patch_size=patch_size)
desc2 = describe_keypoints(img2, keypoints2,
                           desc_func=hog_descriptor,
                           patch_size=patch_size)

# Match descriptors in image1 to those in image2
matches = match_descriptors(desc1, desc2, 0.7)

# Plot matches
fig, ax = plt.subplots(1, 1, figsize=(15, 12))
ax.axis('off')
plot_matches(ax, img1_color, img2_color, keypoints1, keypoints2, matches)
plt.show()


## 3.变换矩阵估计

现在我们已经得到了两幅图像之间匹配关键点的列表。我们将利用这些匹配点来找到一个变换矩阵，该矩阵能够将第二幅图像中的点映射到第一幅图像中的对应坐标上。换句话说，如果图像1中的点 $p_1 = [y_1,x_1]$ 与图像2中的点 $p_2=[y_2, x_2]$ 相匹配，我们需要找到一个仿射变换矩阵 $H$，使得：

$$
\tilde{p_2}H = \tilde{p_1},
$$

其中 $\tilde{p_1}$ 和 $\tilde{p_2}$ 分别是 $p_1$ 和 $p_2$ 的齐次坐标。

需要注意的是，我们可能无法找到一个能将图像2中的每一个点都精确映射到图像1中对应点的变换矩阵 $H$。然而，我们可以通过最小二乘法来估计这个变换矩阵。给定 $N$ 对匹配的关键点对，令 $X_1$ 和 $X_2$ 为 $N \times 3$ 的矩阵，其行分别对应图像1和图像2中匹配关键点的齐次坐标。那么，我们可以通过求解以下最小二乘问题来估计 $H$：

$$
X_2 H = X_1
$$

请在中实现 **`fit_affine_matrix`** 函数。

*-提示：阅读关于 `np.linalg.lstsq` 的[文档](https://docs.scipy.org/doc/numpy/reference/generated/numpy.linalg.lstsq.html)*

In [ ]:
def fit_affine_matrix(p1, p2):
    """ Fit affine matrix such that p2 * H = p1

    Hint:
        You can use np.linalg.lstsq function to solve the problem.

    Args:
        p1: an array of shape (M, P)
        p2: an array of shape (M, P)

    Return:
        H: a matrix of shape (P, P) that transform p2 to p1.
    """

    assert (p1.shape[0] == p2.shape[0]),\
        'Different number of points in p1 and p2'
    p1 = pad(p1)
    p2 = pad(p2)

    ### YOUR CODE HERE

    ### END YOUR CODE

    # Sometimes numerical issues cause least-squares to produce the last
    # column which is not exactly [0, 0, 1]
    H[:,2] = np.array([0, 0, 1])
    return H

In [ ]:
# Sanity check for fit_affine_matrix

# Test inputs
a = np.array([[0.5, 0.1], [0.4, 0.2], [0.8, 0.2]])
b = np.array([[0.3, -0.2], [-0.4, -0.9], [0.1, 0.1]])

H = fit_affine_matrix(b, a)

# Target output
sol = np.array(
    [[1.25, 2.5, 0.0],
     [-5.75, -4.5, 0.0],
     [0.25, -1.0, 1.0]]
)

error = np.sum((H - sol) ** 2)

if error < 1e-5:
    print('Implementation correct!')
else:
    print('There is something wrong.')

在确认你的 `fit_affine_matrix` 函数运行正确之后，运行以下代码将其应用于图像。

图像将被进行变换，img2将被映射到img1上。

In [ ]:
from utils import get_output_space, warp_image

# Extract matched keypoints
p1 = keypoints1[matches[:,0]]
p2 = keypoints2[matches[:,1]]

# Find affine transformation matrix H that maps p2 to p1
H = fit_affine_matrix(p1, p2)

output_shape, offset = get_output_space(img1, [img2], [H])
print("Output shape:", output_shape)
print("Offset:", offset)


# Warp images into output sapce
img1_warped = warp_image(img1_color, np.eye(3), output_shape, offset)
img1_mask = (img1_warped != -1) # Mask == 1 inside the image
img1_warped[~img1_mask] = 0     # Return background values to 0

img2_warped = warp_image(img2_color, H, output_shape, offset)
img2_mask = (img2_warped != -1) # Mask == 1 inside the image
img2_warped[~img2_mask] = 0     # Return background values to 0


# Plot warped images
plt.subplot(1,2,1)
plt.imshow(img1_warped/255)
plt.title('Image 1 warped')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(img2_warped/255)
plt.title('Image 2 warped')
plt.axis('off')

plt.show()

接下来，两幅经过变换的图像会被合并，生成一张全景图。此时得到的全景图效果可能还不理想，但我们稍后将采用其他技术来获得更好的结果。


In [ ]:
merged = img1_warped + img2_warped

# Track the overlap by adding the masks together
overlap = (img1_mask * 1.0 +  # Multiply by 1.0 for bool -> float conversion
           img2_mask)

# Normalize through division by `overlap` - but ensure the minimum is 1
normalized = merged / np.maximum(overlap, 1)
plt.imshow(normalized/255)
plt.axis('off')
plt.show()

## 4. RANSAC

我们可以不直接将所有匹配的关键点对都输入 `fit_affine_matrix` 函数，而是使用 RANSAC（随机抽样一致算法）来仅选择“内点”用于计算变换矩阵。

RANSAC 的步骤如下：
    1. 随机选择一组匹配点对
    2. 计算仿射变换矩阵
    3. 使用给定的阈值找出内点
    4. 重复上述步骤，并保留最大的内点集合
    5. 在所有内点上重新计算最小二乘估计

请在 `panorama.py` 中实现 **`ransac`** 函数，然后运行以下代码生成全景图。你将能看出与不使用 RANSAC 得到的结果之间的区别。

In [ ]:
def ransac(keypoints1, keypoints2, matches, n_iters=200, threshold=20):
    """
    Use RANSAC to find a robust affine transformation

        1. Select random set of matches
        2. Compute affine transformation matrix
        3. Compute inliers
        4. Keep the largest set of inliers
        5. Re-compute least-squares estimate on all of the inliers

    Args:
        keypoints1: M1 x 2 matrix, each row is a point
        keypoints2: M2 x 2 matrix, each row is a point
        matches: N x 2 matrix, each row represents a match
            [index of keypoint1, index of keypoint 2]
        n_iters: the number of iterations RANSAC will run
        threshold: the number of threshold to find inliers

    Returns:
        H: a robust estimation of affine transformation from keypoints2 to
        keypoints 1
    """
    # Copy matches array, to avoid overwriting it
    orig_matches = matches.copy()
    matches = matches.copy()

    N = matches.shape[0]
    print(N)
    n_samples = int(N * 0.2)

    matched1 = pad(keypoints1[matches[:,0]])
    matched2 = pad(keypoints2[matches[:,1]])

    max_inliers = np.zeros(N)
    n_inliers = 0

    # RANSAC iteration start
    ### YOUR CODE HERE

    ### END YOUR CODE
    print(H)
    return H, orig_matches[max_inliers]

In [ ]:

# Set seed to compare output against solution image
np.random.seed(131)

H, robust_matches = ransac(keypoints1, keypoints2, matches, threshold=1)

# Visualize robust matches
fig, ax = plt.subplots(1, 1, figsize=(15, 12))
plot_matches(ax, img1_color, img2_color, keypoints1, keypoints2, robust_matches)
plt.axis('off')
plt.show()

We can now use the tranformation matrix $H$ computed using the robust matches to warp our images and create a better-looking panorama.

In [ ]:
output_shape, offset = get_output_space(img1, [img2], [H])

# Warp images into output sapce
img1_warped = warp_image(img1_color, np.eye(3), output_shape, offset)
img1_mask = (img1_warped != -1) # Mask == 1 inside the image
img1_warped[~img1_mask] = 0     # Return background values to 0

img2_warped = warp_image(img2_color, H, output_shape, offset)
img2_mask = (img2_warped != -1) # Mask == 1 inside the image
img2_warped[~img2_mask] = 0     # Return background values to 0

# Plot warped images
plt.subplot(1,2,1)
plt.imshow(img1_warped/255)
plt.title('Image 1 warped')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(img2_warped/255)
plt.title('Image 2 warped')
plt.axis('off')

plt.show()

In [ ]:
merged = img1_warped + img2_warped

# Track the overlap by adding the masks together
overlap = (img1_mask * 1.0 +  # Multiply by 1.0 for bool -> float conversion
           img2_mask)

# Normalize through division by `overlap` - but ensure the minimum is 1
normalized = merged / np.maximum(overlap, 1)
plt.imshow(normalized/255)
plt.axis('off')
plt.show()

## 5. 更好的图像融合

你会注意到最终全景图的中间区域存在模糊和令人不快的线条。使用一种非常简单的技术——线性融合（linear blending），可以消除全景图中的许多这类伪影。

目前，重叠区域中的所有像素都被赋予相等的权重。然而，由于重叠区域左端和右端的像素能够很好地与另一幅图像中的像素互补，因此可以让它们对最终全景图的贡献更小。

线性融合可以通过以下步骤实现：
    1. 定义用于融合的左右边界
    2. 为图像1定义一个权重矩阵，规则如下：
        - 从输出空间的左边界到左边界限，权重为1
        - 从左边界限到右边界限，权重从1线性递减到0
    3. 为图像2定义一个权重矩阵，规则如下：
        - 从输出空间的右边界到右边界限，权重为1
        - 从左边界限到右边界限，权重从0线性递增到1
    4. 将权重矩阵应用于对应的图像
    5. 将图像合并

In [ ]:
def linear_blend(img1_warped, img2_warped):
    """
    Linearly blend two images (grayscale or color).
    """
    # 1. Create masks for both images
    # For color images, a pixel is valid if ANY channel is not zero (or -1)
    if img1_warped.ndim == 3:
        mask1 = np.any(img1_warped > 0, axis=2)
        mask2 = np.any(img2_warped > 0, axis=2)
    else:
        mask1 = img1_warped > 0
        mask2 = img2_warped > 0

    # 2. Find the intersection (overlap area)
    overlap = mask1 & mask2
    
    # 3. Initialize the weight map
    # The weight map is 2D (H, W)
    weights = np.zeros(mask1.shape)
    
    # Get coordinates of overlap pixels
    indices = np.where(overlap)
    if len(indices[0]) == 0:
        # No overlap, just combine them
        return img1_warped + img2_warped

    # 4. Compute weights based on horizontal distance
    # Assume img1 is on the left and img2 is on the right
    min_c = np.min(indices[1])
    max_c = np.max(indices[1])
    
    # Linear ramp from 1 to 0 across the overlap for img1
    # and 0 to 1 for img2
    weights[overlap] = (max_c - indices[1]) / (max_c - min_c)

    # 5. Apply blending
    # IMPORTANT: Use [..., None] to broadcast 2D weights to 3D color channels
    if img1_warped.ndim == 3:
        # Broadcast weights from (H, W) to (H, W, 1)
        w = weights[..., None]
        
        # Blended regions
        out = np.zeros_like(img1_warped)
        
        # Region only in img1
        out[mask1 & ~overlap] = img1_warped[mask1 & ~overlap]
        # Region only in img2
        out[mask2 & ~overlap] = img2_warped[mask2 & ~overlap]
        # Overlap region (Linear combination)
        out[overlap] = w[overlap] * img1_warped[overlap] + (1 - w[overlap]) * img2_warped[overlap]
    else:
        # Grayscale logic
        out = np.zeros_like(img1_warped)
        out[mask1 & ~overlap] = img1_warped[mask1 & ~overlap]
        out[mask2 & ~overlap] = img2_warped[mask2 & ~overlap]
        out[overlap] = weights[overlap] * img1_warped[overlap] + (1 - weights[overlap]) * img2_warped[overlap]

    return out

In [ ]:
# Merge the warped images using linear blending scheme
merged = linear_blend(img1_warped, img2_warped)

plt.imshow(merged/255)
plt.axis('off')
plt.show()

## 6. 多图像拼接
中实现 **`stitch_multiple_images`** 函数，以拼接一组有序的图像序列。
给定一个由 $m$ 张图像组成的序列（$I_1, I_2,...,I_m$），对每一对相邻的图像计算变换矩阵，该矩阵将点从 $I_{i+1}$ 的坐标系转换到 $I_i$ 的坐标系。然后，选择序列中间的一张图像作为参考图像 $I_{ref}$。我们希望最终的全景图像位于 $I_{ref}$ 的坐标系下。

*-提示：*
- 如果你感到困惑，建议回顾线性代数讲义中关于如何组合多个变换矩阵效果的内容。
- 变换矩阵的逆具有相反的效果。在需要计算矩阵的逆时，请使用 [`numpy.linalg.inv`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.linalg.inv.html) 函数。

In [ ]:
def stitch_multiple_images(imgs, imgs_color, desc_func=hog_descriptor, patch_size=16):
    """
    Args:
        imgs: List of grayscale images (for computation)
        imgs_color: List of color images (for final rendering)
        desc_func: Feature descriptor function
        patch_size: Size of patch for descriptors
    """
    # 1. Detect keypoints (Use grayscale imgs)
    keypoints = []
    for img in imgs:
        kypnts = corner_peaks(harris_corners(img, window_size=3),
                              threshold_rel=0.05,
                              exclude_border=patch_size) # Use patch_size to avoid boundary issues
        keypoints.append(kypnts)

    # 2. Describe keypoints (Use grayscale imgs)
    descriptors = []
    for i, kypnts in enumerate(keypoints):
        desc = describe_keypoints(imgs[i], kypnts,
                                  desc_func=desc_func,
                                  patch_size=patch_size)
        descriptors.append(desc)

    # 3. Match keypoints
    matches = []
    for i in range(len(imgs)-1):
        mtchs = match_descriptors(descriptors[i], descriptors[i+1], 0.7)
        matches.append(mtchs)

    ### YOUR CODE HERE

    # 4. Compute Homographies (Hs)

    # 5. Get output space (Based on grayscale shape, it returns 2D shape)


    # 6. Warp Color Images

    
    ### END YOUR CODE

    # 7. Blend Images (Linear blending supports color if updated)
    panorama = imgs_warped[0]
    for i in range(1, len(imgs)):
        # Ensure linear_blend handles (H, W, 3) with broadcasting
        panorama = linear_blend(panorama, imgs_warped[i])

    return panorama

In [ ]:
# Set seed to compare output against solution
np.random.seed(131)

# Load images to be stitched
img1 = imread('yosemite1.jpg', as_gray=True)
img2 = imread('yosemite2.jpg', as_gray=True)
img3 = imread('yosemite3.jpg', as_gray=True)
img4 = imread('yosemite4.jpg', as_gray=True)

imgs = [img1, img2, img3, img4]

img1_color = imread('yosemite1.jpg')
img2_color = imread('yosemite2.jpg')
img3_color = imread('yosemite3.jpg')
img4_color = imread('yosemite4.jpg')
imgs_color = [img1_color, img2_color, img3_color, img4_color]


# Stitch images together
panorama = stitch_multiple_images(imgs, imgs_color, desc_func=hog_descriptor, patch_size=16)


In [ ]:
# Visualize final panorama image
plt.figure(figsize=(20, 20))
plt.imshow(panorama/255)
plt.axis('off')
plt.show()